# Tóm tắt nội dung văn bản pháp luật

In [3]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import json
import sys
from pymongo import MongoClient
from bson import ObjectId
from collections import defaultdict
from datetime import datetime


# PROJECT_PATH= os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
PROJECT_PATH = "/home/ubuntu/projects/AI/git/users/giangnv/core-law-document-sync"
sys.path.append(PROJECT_PATH)
from constants import LLMsConfig, MongoDBConfig, MongoDBCollectionConfig, MigrateConfig
from core.common.llms import LLMs

# Khởi tạo kết nối MongoDB
client = MongoClient(
    host=MongoDBConfig.HOST,
    port=MongoDBConfig.PORT,
    username=MongoDBConfig.USERNAME,
    password=MongoDBConfig.PASSWORD
)

db = client[MigrateConfig.MIGRATE_CORE_DB]
law_documents_collection = db[MongoDBCollectionConfig.LAW_DOCUMENT_COLLECTION_NAME]
law_articles_collection = db[MongoDBCollectionConfig.LAW_ARTICLE_COLLECTION_NAME]
law_summaries_collection = db[MongoDBCollectionConfig.BIZ_SUMMARY_COLLECTION_NAME]


def get_summary(doc_id, version="v1.0", summary_type="structured"):
    """
    Tóm tắt văn bản pháp luật dựa trên tiêu đề, phần, chương, mục, tiểu mục và tiêu đề các điều.
    """
    if not doc_id:
        return "Không tìm thấy văn bản"

    is_new = True
    
    # Step 1: Tìm kiếm trong cơ sở dữ liệu
    summary = law_summaries_collection.find_one({"doc_id": doc_id, "version": version})
    if summary:
        summary_content = summary.get("summary_content", "Không tìm thấy tóm tắt")
        is_new = False
        return is_new, summary_content
    
    
    # Step 2: Lấy thông tin văn bản gốc
    doc = law_documents_collection.find_one({"doc_id": doc_id})
    if not doc:
        return is_new, f"Không tìm thấy văn bản với doc_id = {doc_id}"        
    doc_title = doc.get("doc_title", "Không có tiêu đề")

    # Step 3: Lấy toàn bộ các điều trong văn bản
    articles = list(law_articles_collection.find({"doc_id": doc_id}).sort("article_index", 1))
    if not articles:
        return is_new, f"Văn bản '{doc_title}' chưa có điều nào."

    # Step 4: Nhóm điều theo cấu trúc phân cấp
    summary_structure = defaultdict(lambda: defaultdict(lambda: defaultdict(lambda: defaultdict(list))))
    for a in articles:
        part = a.get("part", "").strip() or "Không phân phần"
        chapter = a.get("chapter", "").strip() or "Không phân chương"
        section = a.get("section", "").strip() or "Không phân mục"
        sub_section = a.get("sub_section", "").strip() or "Không phân tiểu mục"
        title = a.get("article_title", "").strip()

        summary_structure[part][chapter][section][sub_section].append(title)

    # Step 5: Tạo bản tóm tắt có định dạng rõ ràng
    summary_lines = [f"{doc_title}\n"]
    for part, chapters in summary_structure.items():
        if part != "Không phân phần":
            summary_lines.append(f"== {part} ==")
        for chapter, sections in chapters.items():
            if chapter != "Không phân chương":
                summary_lines.append(f"  • {chapter}")
            for section, sub_sections in sections.items():
                if section != "Không phân mục":
                    summary_lines.append(f"    ◦ {section}")
                for sub_section, articles_list in sub_sections.items():                    
                    if sub_section != "Không phân tiểu mục":
                        summary_lines.append(f"      ▪ {sub_section}")
                    for article_title in articles_list:
                        summary_lines.append(f"        - {article_title}")
    summary = "\n".join(summary_lines)
    
    # Step 6: Lưu tóm tắt vào cơ sở dữ liệu
    law_summaries_collection.insert_one({
        "doc_id": doc_id,
        "version": version,
        "summary_type": summary_type,
        "summary_content": summary,
        "created_at": datetime.now(),
        "last_modified": datetime.now()
    })
    return is_new, summary

In [ ]:
# content = get_summary("100704")

# Training phân loại tài liệu

In [ ]:
import os
import json
import random
import fasttext
import uuid
import sys
from collections import defaultdict
from datetime import datetime
from pymongo import MongoClient
from bson import ObjectId
from sklearn.metrics import classification_report


# PROJECT_PATH = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
PROJECT_PATH = "/home/ubuntu/projects/AI/git/users/giangnv/core-law-document-sync"
sys.path.append(PROJECT_PATH)
from constants import LLMsConfig, MongoDBConfig, MongoDBCollectionConfig, MigrateConfig
from core.common.llms import LLMs

# Khởi tạo kết nối MongoDB
client = MongoClient(
    host=MongoDBConfig.HOST,
    port=MongoDBConfig.PORT,
    username=MongoDBConfig.USERNAME,
    password=MongoDBConfig.PASSWORD
)

db = client[MigrateConfig.MIGRATE_CORE_DB]
law_documents_collection = db[MongoDBCollectionConfig.LAW_DOCUMENT_COLLECTION_NAME]
law_articles_collection = db[MongoDBCollectionConfig.LAW_ARTICLE_COLLECTION_NAME]
law_summaries_collection = db[MongoDBCollectionConfig.BIZ_SUMMARY_COLLECTION_NAME]
law_tree_components = db[MongoDBCollectionConfig.LAW_TREE_COMPONENT_COLLECTION_NAME]



CONFIG = {
    "lr": 0.1,
    "epoch": 1000,
    "dim": 512,
    "wordngrams": 3
}

# =========================
# 0 LOAD TREE
# =========================
def get_classes(tree_id: str):
    """
    Lấy danh sách các chủ đề con (subject_level='CHILD') trong cây chuyên đề.
    
    Args:
        tree_id (str): ID của cây chuyên đề.
        db: MongoDB database instance.
    
    Returns:
        List[dict]: Danh sách gồm các phần tử dạng:
            {
                "subject_name": str,
                "doc_id_includes": list[str]
            }
    """
    # Truy vấn tất cả bản ghi con của cây
    children = law_tree_components.find({
        "tree_id": tree_id,
        "subject_level": "CHILD"
    })

    # Chuyển đổi kết quả sang danh sách gọn gàng
    classes = []
    for child in children:
        classes.append({
            "class_name": child.get("subject_name", "").strip(),
            "doc_ids": child.get("doc_id_includes", [])
        })
    return classes   


# =========================
# 1️⃣ LOAD DATA
# =========================
def load_data_summary(class_name: str, doc_ids: list):
    """
    Tải dữ liệu từ law_summaries_collection trong MongoDB.
    Trả về list các tuple (text, label)
    """
    data = []
    for doc_id in doc_ids:
        _, summary_content = get_summary(doc_id)
        if summary_content:
            data.append((summary_content, class_name))
    return data


# =========================
# 2️⃣ TẠO DATASET FASTTEXT
# =========================
def prepare_fasttext_datasets(data, output_dir="datasets", test_ratio=0.2):
    """
    Tạo file train/test theo định dạng FastText
    """
    os.makedirs(output_dir, exist_ok=True)
    random.shuffle(data)
    split_idx = int(len(data) * (1 - test_ratio))
    train_data, test_data = data[:split_idx], data[split_idx:]

    train_path = os.path.join(output_dir, "train.txt")
    test_path = os.path.join(output_dir, "test.txt")

    with open(train_path, "w", encoding="utf-8") as f_train:
        for text, label in train_data:
            label = label.replace(' ', '__')            
            text = text.replace("\n", " ")
            f_train.write(f"__label__{label} {text}\n")

    with open(test_path, "w", encoding="utf-8") as f_test:
        for text, label in test_data:
            label = label.replace(' ', '__')
            text = text.replace("\n", " ")
            f_test.write(f"__label__{label} {text}\n")
    return train_path, test_path


# =========================
# 3️⃣ TRAINING
# =========================
def train_fasttext_model(train_file, test_file, output_dir="models", model_name="law_classifier", lr=0.1, epoch=30, dim=128, wordNgrams=2):
    """
    Huấn luyện mô hình FastText và lưu kết quả
    """
    os.makedirs(output_dir, exist_ok=True)
    model_path = os.path.join(output_dir, f"{model_name}.bin")
    metrics_path = os.path.join(output_dir, f"{model_name}_metrics.json")

    model = fasttext.train_supervised(
        input=train_file,
        lr=lr,
        epoch=epoch,
        dim=dim,
        wordNgrams=wordNgrams,
        loss="softmax"
    )

    # Lưu model
    model.save_model(model_path)

    # Đánh giá
    n_examples, precision, recall  = model.test(test_file)
    f1 = 2 * precision * recall / (precision + recall + 1e-9)

    metrics = {
        "precision": round(precision, 4),
        "recall": round(recall, 4),
        "f1_score": round(f1, 4),
        "n_samples": n_examples,
        "trained_at": datetime.utcnow().isoformat()
    }

    with open(metrics_path, "w", encoding="utf-8") as f:
        json.dump(metrics, f, ensure_ascii=False, indent=2)


    return model_path, metrics_path


# =========================
# 4️⃣ INFERENCE
# =========================

def predict_text(model_path, text):
    """
    Dự đoán nhãn cho đoạn văn bản.
    """
    model = fasttext.load_model(model_path)
    result = model.predict(text)
    label, prob = model.predict(text)
    return label[0].replace("__label__", ""), prob[0]


# =========================
# 5️⃣ FULL PIPELINE
# =========================
def train(tree_id, config=CONFIG):
    """
    Full pipeline: load -> split -> train -> test -> save -> return model path
    """

    # 0. Load classes
    classes = get_classes(tree_id)
    
    # 1. Load dữ liệu
    final_data = []
    for _class in classes:    
        class_name = _class.get("class_name")
        doc_ids = _class.get("doc_ids")           
        try:
            data = load_data_summary(class_name, doc_ids)
        except Exception as e:
            raise e
            continue    
        final_data.extend(data)
    
    
    # 2. Tạo train/test file
    folder_name = str(uuid.uuid4())
    OUTPUT_FOLDER = os.path.join("./datasets", folder_name)
    if not os.path.exists(OUTPUT_FOLDER):
        os.makedirs(OUTPUT_FOLDER)
    TEST_RATIO = 0.2
    train_path, test_path = prepare_fasttext_datasets(data=final_data, output_dir=OUTPUT_FOLDER, test_ratio=TEST_RATIO)


    # 3. Huấn luyện model
    model_path, metrics_path = train_fasttext_model(train_file=train_path, 
                                                    test_file=test_path, 
                                                    output_dir=OUTPUT_FOLDER, 
                                                    lr=config.get("lr", 0.1), 
                                                    epoch=config.get("epoch", 1000), 
                                                    dim=config.get("dim", 512), 
                                                    wordNgrams=config.get("wordngrams", 3))
    return model_path, metrics_path


# 0. Load classes from tree

In [7]:
TREE_ID = "2037c7dd-5d6d-4a5c-ba65-7d52e4f697ba"
classes = get_classes(tree_id=TREE_ID)

# 1. Load Data Summary

In [ ]:
final_data = []
for _class in classes:    
    class_name = _class.get("class_name")
    doc_ids = _class.get("doc_ids")    
    
    try:
        data = load_data_summary(class_name, doc_ids)
    except Exception as e:
        raise e
        continue    
    final_data.extend(data)

Class: An ninh quốc gia
Number of documents: 11
Class: Bảo vệ bí mật nhà nước
Number of documents: 9
Class: Bảo vệ công trình quan trọng liên quan đến an ninh quốc gia
Number of documents: 3
Class: Biên giới quốc gia
Number of documents: 18
Class: Biển Việt Nam
Number of documents: 5
Class: Công an nhân dân
Number of documents: 21
Class: Cơ yếu
Number of documents: 7
Class: Nhập cảnh, xuất cảnh, quá cảnh, cư trú của người nước ngoài tại Việt Nam
Number of documents: 18
Class: Phòng, chống khủng bố
Number of documents: 6
Class: Xuất cảnh, nhập cảnh của công dân Việt Nam
Number of documents: 11
Class: An ninh mạng
Number of documents: 2
Class: Cảnh vệ
Number of documents: 5
Class: Bảo hiểm y tế
Number of documents: 34
Class: An toàn thông tin mạng
Number of documents: 15
Class: Bưu chính
Number of documents: 27
Class: Công nghệ thông tin
Number of documents: 72
Class: Tần số vô tuyến điện
Number of documents: 30
Class: Đấu giá tài sản
Number of documents: 11
Class: Công chứng
Number of d

# 2 TẠO DATASET FASTTEXT

In [9]:
folder_name = "version_1"
OUTPUT_FOLDER = os.path.join("./datasets", folder_name)
if not os.path.exists(OUTPUT_FOLDER):
    os.makedirs(OUTPUT_FOLDER)
TEST_RATIO = 0.2

train_path, test_path = prepare_fasttext_datasets(data=final_data, output_dir=OUTPUT_FOLDER, test_ratio=TEST_RATIO)

# 3 Training Model

In [16]:
train_path = "./datasets/version_1/train.txt"
test_path = "./datasets/version_1/test.txt"

LR = 0.1
EPOCH = 1000
DIM = 512
WORDNGRAMS = 3
 
model_path, metrics_path = train_fasttext_model(train_file=train_path, test_file=test_path, output_dir=OUTPUT_FOLDER, lr=LR, epoch=EPOCH, dim=DIM, wordNgrams=WORDNGRAMS)

🚀 Bắt đầu huấn luyện mô hình...


Read 1M words
Number of words:  15990
Number of labels: 229
Progress: 100.0% words/sec/thread:  171647 lr:  0.000000 avg.loss:  0.417947 ETA:   0h 0m 0s words/sec/thread:  171869 lr:  0.094105 avg.loss:  3.989841 ETA:   0h 4m 4s  0h 4m 2s ETA:   0h 4m 0s  8.3% words/sec/thread:  172426 lr:  0.091653 avg.loss:  3.309727 ETA:   0h 3m57s  0h 3m52s 173018 lr:  0.089565 avg.loss:  2.845819 ETA:   0h 3m51s 3m40s ETA:   0h 3m37s 3m32s 28.1% words/sec/thread:  173065 lr:  0.071887 avg.loss:  1.244681 ETA:   0h 3m 5s avg.loss:  1.180504 ETA:   0h 3m 0s% words/sec/thread:  173057 lr:  0.068508 avg.loss:  1.131200 ETA:   0h 2m56s% words/sec/thread:  173276 lr:  0.065939 avg.loss:  1.060795 ETA:   0h 2m49sh 2m46s 173432 lr:  0.061897 avg.loss:  0.962051 ETA:   0h 2m39s 0.061818 avg.loss:  0.960057 ETA:   0h 2m39s 173406 lr:  0.060708 avg.loss:  0.932777 ETA:   0h 2m36s% words/sec/thread:  173280 lr:  0.057404 avg.loss:  0.868634 ETA:   0h 2m27s avg.loss:  0.847967 ETA:   0h 2m25s words/sec/thread:

✅ Đã lưu mô hình tại: ./datasets/version_1/law_classifier.bin
📊 Kết quả đánh giá: {'precision': 0.6966, 'recall': 0.6966, 'f1_score': 0.6966, 'n_samples': 1101, 'trained_at': '2025-10-30T03:57:01.799810'}


In [12]:
# %pip install numpy==1.26.4

In [14]:
text =  """Quyết định 30/2008/QĐ-BGDĐT về tổ chức đào tạo, bồi dưỡng, kiểm tra và cấp chứng chỉ ngoại ngữ, tin học theo chương trình giáo dục thường xuyên do Bộ trưởng Bộ Giáo dục và Đào tạo ban hành' chưa có điều nào."""
predict_text(model_path, text)

result: (('__label__Giáo__dục',), array([0.7977711]))


('Giáo__dục', 0.7977710962295532)

# Data

In [ ]:
import os
import json
import sys
from pymongo import MongoClient
from bson import ObjectId
from collections import defaultdict
from datetime import datetime


# PROJECT_PATH= os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
PROJECT_PATH = "/home/ubuntu/projects/AI/git/users/giangnv/core-law-document-sync"
sys.path.append(PROJECT_PATH)
from constants import LLMsConfig, MongoDBConfig, MongoDBCollectionConfig, MigrateConfig
from core.common.llms import LLMs

# Khởi tạo kết nối MongoDB
client = MongoClient(
    host=MongoDBConfig.HOST,
    port=MongoDBConfig.PORT,
    username=MongoDBConfig.USERNAME,
    password=MongoDBConfig.PASSWORD
)

db = client[MigrateConfig.MIGRATE_CORE_DB]
law_documents_collection = db[MongoDBCollectionConfig.LAW_DOCUMENT_COLLECTION_NAME]
law_articles_collection = db[MongoDBCollectionConfig.LAW_ARTICLE_COLLECTION_NAME]
law_summaries_collection = db[MongoDBCollectionConfig.BIZ_SUMMARY_COLLECTION_NAME]